In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

from ADFWI.utils.assessment_metric import MAPE, MSE, SNR, SSIM

base_path = "/home/bingxing2/ailab/scxlab0055/project/04_Inversion/ADFWI-github/examples/DR-FWI/Inductive_bias/regularization/Marmousi2-nowater-RealNoise/RealNoiseIRIS-Smooth=6/snr=0.3"

init_model = np.load(os.path.join(base_path,"model/init_model.npz"))
true_model = np.load(os.path.join(base_path,"model/true_model.npz"))
init_v   = init_model["vp"]
init_rho = init_model["rho"]
true_v   = true_model["vp"]
true_rho = true_model["rho"]

ox, oz  = 0, 0        
nz, nx  = 76, 200      
dx, dz  = 40, 40         
nt, dt  = 2500, 0.003     
nabc    = 30                 
x       = np.arange(nx)*dx/1000
z       = np.arange(nz)*dz/1000
x_mesh,z_mesh = np.meshgrid(x,z)
src_z = np.array([1  for i in range(2,nx-1,5)])*dz/1000
src_x = np.array([i  for i in range(2,nx-1,5)])*dx/1000
rcv_z = np.array([1  for i in range(0,nx,1)])*dz/1000
rcv_x = np.array([j  for j in range(0,nx,1)])*dz/1000
vmin = true_v.min();vmax = true_v.max()  

MAX_ITER = 500

In [ ]:
itervp_baseline      = np.load(os.path.join(base_path,"inversion-vp-baseline/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x64      = np.load(os.path.join(base_path,"inversion-vp-CNN-1x64-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x128     = np.load(os.path.join(base_path,"inversion-vp-CNN-1x128-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x256     = np.load(os.path.join(base_path,"inversion-vp-CNN-1x256-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x64      = np.load(os.path.join(base_path,"inversion-vp-CNN-2x64-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x128     = np.load(os.path.join(base_path,"inversion-vp-CNN-2x128-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x256     = np.load(os.path.join(base_path,"inversion-vp-CNN-2x256-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x64      = np.load(os.path.join(base_path,"inversion-vp-CNN-3x64-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x128     = np.load(os.path.join(base_path,"inversion-vp-CNN-3x128-v/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x256     = np.load(os.path.join(base_path,"inversion-vp-CNN-3x256-v/iter_vp.npz"))["data"][:MAX_ITER]


iterloss_baseline    = np.load(os.path.join(base_path,"inversion-vp-baseline/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_CNN_1x64    = np.load(os.path.join(base_path,"inversion-vp-CNN-1x64-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_CNN_1x128    = np.load(os.path.join(base_path,"inversion-vp-CNN-1x128-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_CNN_1x256    = np.load(os.path.join(base_path,"inversion-vp-CNN-1x256-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_CNN_2x64    = np.load(os.path.join(base_path,"inversion-vp-CNN-2x64-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_CNN_2x128    = np.load(os.path.join(base_path,"inversion-vp-CNN-2x128-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_CNN_2x256    = np.load(os.path.join(base_path,"inversion-vp-CNN-2x256-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_CNN_3x64    = np.load(os.path.join(base_path,"inversion-vp-CNN-3x64-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_CNN_3x128    = np.load(os.path.join(base_path,"inversion-vp-CNN-3x128-v/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_CNN_3x256    = np.load(os.path.join(base_path,"inversion-vp-CNN-3x256-v/iter_loss.npz"))["data"][:MAX_ITER]


In [ ]:
import matplotlib.transforms as mtransforms
from scipy.interpolate import griddata

def plot_vel_single_for_all(fig,ax,v,title="",MSE="",vmin=None,vmax=None,cmap = "rainbow"):
    plt.rc('font',family='Times New Roman')
    # plm = ax.pcolormesh(x_mesh, z_mesh, v,cmap=cmap,vmin=vmin,vmax=vmax)
    x = np.arange(nx*3)*dx/3/1000
    z = np.arange(nz*3)*dz/3/1000
    x_mesh_new,z_mesh_new = np.meshgrid(x,z)

    v_new = griddata((x_mesh.flatten(), z_mesh.flatten()), v.flatten(), (x_mesh_new, z_mesh_new), method='cubic')
    
    plm = ax.pcolormesh(x_mesh_new, z_mesh_new, v_new,cmap=cmap,vmin=vmin,vmax=vmax,shading="nearest")
    ax.invert_yaxis()
    ax.tick_params(labelsize = 14)
    ax.set_title(title,fontsize=14)
    ax.text(0.2,0.4,MSE,fontsize=14,c="w")
    return plm

def add_right_cax(ax, pad, width):
    axpos = ax.get_position()
    caxpos = mtransforms.Bbox.from_extents(
        axpos.x1 + pad,
        axpos.y0,
        axpos.x1 + pad + width,
        axpos.y1
    )
    cax = ax.figure.add_axes(caxpos)

    return cax

def add_bottom_cax(ax, pad, height):
    axpos = ax.get_position()
    caxpos = mtransforms.Bbox.from_extents(
        axpos.x0,
        axpos.y0 - pad - height,
        axpos.x1,
        axpos.y0 - pad
    )
    cax = ax.figure.add_axes(caxpos)
    
    return cax

def plot_vel_singleline_for_all(ax,v_true,v_init,v_inv,x_distance,title,show_xlabel=True,show_ylabel=False,show_legend=False):
    ax.plot(v_true[:,int(x_distance//dx)]/1000,  z, c='k',   linewidth=2, linestyle="-" ,label="True")
    ax.plot(v_init[:,int(x_distance//dx)]/1000,  z, c='gray',linewidth=2, linestyle="-" ,label="Init")
    ax.plot(v_inv [:,int(x_distance//dx)]/1000,  z, c='r',   linewidth=2, linestyle="--",label="Inverted")
    ax.tick_params(labelsize = 12)
    if not show_xlabel:
        ax.set_xticks([])

    if not show_ylabel:
        ax.set_yticks([])
    else:
        ax.tick_params(labelsize = 10)
    ax.invert_yaxis()
    
    if show_legend:
        ax.legend(fontsize = 12)
    ax.set_title(title,fontsize=12)

In [ ]:

fig,axs = plt.subplots(4,2,figsize=(13,10))
im = plot_vel_single_for_all(fig,axs[0][0],true_v    ,title="True Model")
axs[0][0].scatter(rcv_x[1::2],rcv_z[1::2]+0.05,c="w",marker="v",s=10)
axs[0][0].scatter(src_x[1::2],src_z[1::2]+0.05,c="r",marker="*",s=60)
axs[0][0].set_ylabel("Depth (km)",fontsize=15)
axs[0][0].set_ylabel("Depth (km)",fontsize=15)
plot_vel_single_for_all(fig,axs[0][1],itervp_baseline[-1]   ,title=r"Baseline"  ,MSE="MAPE:{:.2f}".format(MAPE(true_v,itervp_baseline[-1])))

plot_vel_single_for_all(fig,axs[1][0],itervp_CNN_1x64[-1]   ,title=r"CNN-1x64"  ,MSE="MAPE:{:.2f}".format(MAPE(true_v,itervp_CNN_1x64[-1])))
axs[1][0].set_ylabel("Depth (km)",fontsize=15)
plot_vel_single_for_all(fig,axs[1][1],itervp_CNN_1x128[-1]  ,title=r"CNN-1x128" ,MSE="MAPE:{:.2f}".format(MAPE(true_v,itervp_CNN_1x128[-1])))

plot_vel_single_for_all(fig,axs[2][0],itervp_CNN_1x256[-1]  ,title=r"CNN-1x256" ,MSE="MAPE:{:.2f}".format(MAPE(true_v,itervp_CNN_1x256[-1])))
axs[2][0].set_ylabel("Depth (km)",fontsize=15)
plot_vel_single_for_all(fig,axs[2][1],itervp_CNN_3x64[-1]   ,title=r"CNN-3x64"  ,MSE="MAPE:{:.2f}".format(MAPE(true_v,itervp_CNN_3x64[-1])))

plot_vel_single_for_all(fig,axs[3][0],itervp_CNN_3x128[-1]  ,title=r"CNN-3x128" ,MSE="MAPE:{:.2f}".format(MAPE(true_v,itervp_CNN_3x128[-1])))
axs[3][0].set_ylabel("Depth (km)",fontsize=15)
plot_vel_single_for_all(fig,axs[3][1],itervp_CNN_3x256[-1]  ,title=r"CNN-3x256" ,MSE="MAPE:{:.2f}".format(MAPE(true_v,itervp_CNN_3x256[-1])))

axs[0][0].set_xticks([])
axs[0][1].set_xticks([])
axs[0][1].set_yticks([])

axs[1][0].set_xticks([])
axs[1][1].set_xticks([])
axs[1][1].set_yticks([])

axs[2][0].set_xticks([])
axs[2][1].set_xticks([])
axs[2][1].set_yticks([])

axs[3][1].set_yticks([])

from matplotlib.ticker import MaxNLocator
axs[0][0].yaxis.set_major_locator(MaxNLocator(5))
axs[1][0].yaxis.set_major_locator(MaxNLocator(5))
axs[2][0].yaxis.set_major_locator(MaxNLocator(5))
axs[3][0].yaxis.set_major_locator(MaxNLocator(5))
axs[3][0].xaxis.set_major_locator(MaxNLocator(7))
axs[3][1].xaxis.set_major_locator(MaxNLocator(7))

axs[3][0].set_xlabel("Distance (km)", fontsize=15)
axs[3][1].set_xlabel("Distance (km)", fontsize=15)

plt.subplots_adjust(hspace=0.2,wspace=0.03)

cbar = fig.colorbar(im, ax=axs,orientation='vertical',pad = 0.02,shrink=0.6)
cbar.ax.set_title("(m/s)",fontsize=15,loc='center')
cbar.ax.tick_params(labelsize=15)

axs[0][0].text(0.01, 0.05, "(a)", transform=axs[0][0].transAxes, fontsize=15, verticalalignment='bottom', horizontalalignment='left',c='k')
axs[0][1].text(0.01, 0.05, "(b)", transform=axs[0][1].transAxes, fontsize=15, verticalalignment='bottom', horizontalalignment='left',c='k')
axs[1][0].text(0.01, 0.05, "(c)", transform=axs[1][0].transAxes, fontsize=15, verticalalignment='bottom', horizontalalignment='left',c='k')
axs[1][1].text(0.01, 0.05, "(d)", transform=axs[1][1].transAxes, fontsize=15, verticalalignment='bottom', horizontalalignment='left',c='k')
axs[2][0].text(0.01, 0.05, "(e)", transform=axs[2][0].transAxes, fontsize=15, verticalalignment='bottom', horizontalalignment='left',c='k')
axs[2][1].text(0.01, 0.05, "(f)", transform=axs[2][1].transAxes, fontsize=15, verticalalignment='bottom', horizontalalignment='left',c='k')
axs[3][0].text(0.01, 0.05, "(g)", transform=axs[3][0].transAxes, fontsize=15, verticalalignment='bottom', horizontalalignment='left',c='k')
axs[3][1].text(0.01, 0.05, "(h)", transform=axs[3][1].transAxes, fontsize=15, verticalalignment='bottom', horizontalalignment='left',c='k')

# plt.savefig("/ailab/user/liufeng1/project/04_Inversion/ADFWI-github/examples/dip/reparameterization-strategy/Marmousi2-nowater/cmp/Figures/Architecture_and_Strategy_Marmousi2_model.png",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Define color scheme
# 优化后的颜色方案，增强区分度和美观性
colors = {
    "baseline": "black",

    # CNN1 - 绿色系
    "CNN1x64":  "#66c2a5",   # 淡绿
    "CNN1x128": "#1b9e77",   # 中绿
    "CNN1x256": "#00441b",   # 深绿

    # CNN2 - 蓝色系
    "CNN2x64":  "#8da0cb",   # 淡蓝
    "CNN2x128": "#377eb8",   # 中蓝
    "CNN2x256": "#081d58",   # 深蓝

    # CNN3 - 红色系
    "CNN3x64":  "#fc8d62",   # 淡红
    "CNN3x128": "#e41a1c",   # 中红
    "CNN3x256": "#67000d",   # 深红

    # CNN4 - 紫色系
    "CNN4x64":  "#e78ac3",   # 淡紫
    "CNN4x128": "#984ea3",   # 中紫
    "CNN4x256": "#4d004b"    # 深紫
}


# Set up the figure
plt.figure(figsize=(12, 8))

# First subplot: Data Residual
ax1 = plt.subplot(221)

# Plot original curves
ax1.plot(iterloss_baseline, label="Baseline"  , color=colors["baseline"] , linestyle='-', linewidth=2)
ax1.plot(iterloss_CNN_1x64,   label="CNN-1x64"  , color=colors["CNN1x64"]  , linestyle="-", linewidth=2)
ax1.plot(iterloss_CNN_1x128,  label="CNN-1x128" , color=colors["CNN1x128"] , linestyle="-", linewidth=2)
ax1.plot(iterloss_CNN_1x256,  label="CNN-1x256" , color=colors["CNN1x256"] , linestyle="-", linewidth=2)
ax1.plot(iterloss_CNN_2x64,   label="CNN-2x64"  , color=colors["CNN2x64"]  , linestyle="-", linewidth=2)
ax1.plot(iterloss_CNN_2x128,  label="CNN-2x128" , color=colors["CNN2x128"] , linestyle="-", linewidth=2)
ax1.plot(iterloss_CNN_2x256,  label="CNN-2x256" , color=colors["CNN2x256"] , linestyle="-", linewidth=2)
ax1.plot(iterloss_CNN_3x64,   label="CNN-3x64"  , color=colors["CNN3x64"]  , linestyle="-", linewidth=2)
ax1.plot(iterloss_CNN_3x128,  label="CNN-3x128" , color=colors["CNN3x128"] , linestyle="-", linewidth=2)
ax1.plot(iterloss_CNN_3x256,  label="CNN-3x256" , color=colors["CNN3x256"] , linestyle="-", linewidth=2)

# Improve axis style
ax1.set_ylabel('Global-Correlation Misfits', fontsize=12)
# ax1.set_title("Data Residual", fontsize=12)
ax1.set_xticklabels([])  # Hide x-axis labels
ax1.tick_params(axis="x", which="both", length=0)
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend(loc='upper right', fontsize=10)

# Create inset zoom-in (focus on x: 200-300, y: -8200 to -7500)
axins = ax1.inset_axes([0.25, 0.4, 0.4, 0.3])  # (x_pos, y_pos, width, height)
axins.set_xlim(400, 500)
axins.set_ylim(-2100, -2400)

# Plot the zoomed-in curves
axins.plot(iterloss_baseline, color=colors["baseline"], linestyle='-', linewidth=2)
axins.plot(iterloss_CNN_1x64  , color=colors["CNN1x64"] , linestyle="-", linewidth=2)
axins.plot(iterloss_CNN_1x128 , color=colors["CNN1x128"], linestyle="-", linewidth=2)
axins.plot(iterloss_CNN_1x256 , color=colors["CNN1x256"], linestyle="-", linewidth=2)
axins.plot(iterloss_CNN_2x64  , color=colors["CNN2x64"] , linestyle="-", linewidth=2)
axins.plot(iterloss_CNN_2x128 , color=colors["CNN2x128"], linestyle="-", linewidth=2)
axins.plot(iterloss_CNN_2x256 , color=colors["CNN2x256"], linestyle="-", linewidth=2)
axins.plot(iterloss_CNN_3x64  , color=colors["CNN3x64"] , linestyle="-", linewidth=2)
axins.plot(iterloss_CNN_3x128 , color=colors["CNN3x128"], linestyle="-", linewidth=2)
axins.plot(iterloss_CNN_3x256 , color=colors["CNN3x256"], linestyle="-", linewidth=2)

# Remove ticks from inset
axins.set_xticks([])
axins.set_yticks([])

# Indicate zoom-in region
ax1.indicate_inset_zoom(axins, edgecolor="black")

#######################################################
# Second subplot: Model Residual
ax2 = plt.subplot(222)
vp_init_model_loss      =           MAPE(true_v,init_v)
vp_baseline_loss        = np.insert(MAPE(true_v,itervp_baseline),0,vp_init_model_loss)
vp_CNN_1x64_loss_vel    = np.insert(MAPE(true_v,itervp_CNN_1x64),0,vp_init_model_loss)
vp_CNN_1x128_loss_vel   = np.insert(MAPE(true_v,itervp_CNN_1x128),0,vp_init_model_loss)
vp_CNN_1x256_loss_vel   = np.insert(MAPE(true_v,itervp_CNN_1x256),0,vp_init_model_loss)
vp_CNN_2x64_loss_vel    = np.insert(MAPE(true_v,itervp_CNN_2x64),0,vp_init_model_loss)
vp_CNN_2x128_loss_vel   = np.insert(MAPE(true_v,itervp_CNN_2x128),0,vp_init_model_loss)
vp_CNN_2x256_loss_vel   = np.insert(MAPE(true_v,itervp_CNN_2x256),0,vp_init_model_loss)
vp_CNN_3x64_loss_vel    = np.insert(MAPE(true_v,itervp_CNN_3x64),0,vp_init_model_loss)
vp_CNN_3x128_loss_vel   = np.insert(MAPE(true_v,itervp_CNN_3x128),0,vp_init_model_loss)
vp_CNN_3x256_loss_vel   = np.insert(MAPE(true_v,itervp_CNN_3x256),0,vp_init_model_loss)
ax2.plot(vp_baseline_loss, label="Baseline", color=colors["baseline"], linestyle='-', linewidth=2)
ax2.plot(vp_CNN_1x64_loss_vel, label="CNN-1x64", color=colors["CNN1x64"], linestyle="-", linewidth=2)
ax2.plot(vp_CNN_1x128_loss_vel, label="CNN-1x128", color=colors["CNN1x128"], linestyle="-", linewidth=2)
ax2.plot(vp_CNN_1x256_loss_vel, label="CNN-1x256", color=colors["CNN1x256"], linestyle="-", linewidth=2)
ax2.plot(vp_CNN_2x64_loss_vel, label="CNN-2x64", color=colors["CNN2x64"], linestyle="-", linewidth=2)
ax2.plot(vp_CNN_2x128_loss_vel, label="CNN-2x128", color=colors["CNN2x128"], linestyle="-", linewidth=2)
ax2.plot(vp_CNN_2x256_loss_vel, label="CNN-2x256", color=colors["CNN2x256"], linestyle="-", linewidth=2)
ax2.plot(vp_CNN_3x64_loss_vel, label="CNN-3x64", color=colors["CNN3x64"], linestyle="-", linewidth=2)
ax2.plot(vp_CNN_3x128_loss_vel, label="CNN-3x128", color=colors["CNN3x128"], linestyle="-", linewidth=2)
ax2.plot(vp_CNN_3x256_loss_vel, label="CNN-3x256", color=colors["CNN3x256"], linestyle="-", linewidth=2)

# Improve axis style
ax2.set_ylabel('MAPE', fontsize=12)
# ax2.set_title("Model Residual", fontsize=12)
ax2.set_xticklabels([])  # Hide x-axis labels
ax2.tick_params(axis="x", which="both", length=0)
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend(loc='upper right', fontsize=10)


#######################################################
# Second subplot: Model Residual
ax3 = plt.subplot(223)
vp_init_model_loss      =           SNR(true_v,init_v)
vp_baseline_loss        = np.insert(SNR(true_v,itervp_baseline),0,vp_init_model_loss)
vp_CNN_1x64_loss_vel    = np.insert(SNR(true_v,itervp_CNN_1x64),0,vp_init_model_loss)
vp_CNN_1x128_loss_vel   = np.insert(SNR(true_v,itervp_CNN_1x128),0,vp_init_model_loss)
vp_CNN_1x256_loss_vel   = np.insert(SNR(true_v,itervp_CNN_1x256),0,vp_init_model_loss)
vp_CNN_2x64_loss_vel    = np.insert(SNR(true_v,itervp_CNN_2x64),0,vp_init_model_loss)
vp_CNN_2x128_loss_vel   = np.insert(SNR(true_v,itervp_CNN_2x128),0,vp_init_model_loss)
vp_CNN_2x256_loss_vel   = np.insert(SNR(true_v,itervp_CNN_2x256),0,vp_init_model_loss)
vp_CNN_3x64_loss_vel    = np.insert(SNR(true_v,itervp_CNN_3x64),0,vp_init_model_loss)
vp_CNN_3x128_loss_vel   = np.insert(SNR(true_v,itervp_CNN_3x128),0,vp_init_model_loss)
vp_CNN_3x256_loss_vel   = np.insert(SNR(true_v,itervp_CNN_3x256),0,vp_init_model_loss)
ax3.plot(vp_baseline_loss, label="Baseline", color=colors["baseline"], linestyle='-', linewidth=2)
ax3.plot(vp_CNN_1x64_loss_vel, label="CNN-1x64", color=colors["CNN1x64"], linestyle="-", linewidth=2)
ax3.plot(vp_CNN_1x128_loss_vel, label="CNN-1x128", color=colors["CNN1x128"], linestyle="-", linewidth=2)
ax3.plot(vp_CNN_1x256_loss_vel, label="CNN-1x256", color=colors["CNN1x256"], linestyle="-", linewidth=2)
ax3.plot(vp_CNN_2x64_loss_vel, label="CNN-2x64", color=colors["CNN2x64"], linestyle="-", linewidth=2)
ax3.plot(vp_CNN_2x128_loss_vel, label="CNN-2x128", color=colors["CNN2x128"], linestyle="-", linewidth=2)
ax3.plot(vp_CNN_2x256_loss_vel, label="CNN-2x256", color=colors["CNN2x256"], linestyle="-", linewidth=2)
ax3.plot(vp_CNN_3x64_loss_vel, label="CNN-3x64", color=colors["CNN3x64"], linestyle="-", linewidth=2)
ax3.plot(vp_CNN_3x128_loss_vel, label="CNN-3x128", color=colors["CNN3x128"], linestyle="-", linewidth=2)
ax3.plot(vp_CNN_3x256_loss_vel, label="CNN-3x256", color=colors["CNN3x256"], linestyle="-", linewidth=2)

# Improve axis style for second subplot
ax3.set_xlabel('Iterations', fontsize=12)
ax3.set_ylabel('SNR', fontsize=12)
# ax3.set_title("Model Residual", fontsize=12)
ax3.grid(True, linestyle='--', alpha=0.6)

#######################################################
# Second subplot: Model Residual
WIN_SIZE=3
ax4 = plt.subplot(224)
vp_init_model_loss      =           SSIM(true_v,init_v,win_size=WIN_SIZE)
vp_baseline_loss        = np.insert(SSIM(true_v,itervp_baseline,win_size=WIN_SIZE),0,vp_init_model_loss)
vp_CNN_1x64_loss_vel    = np.insert(SSIM(true_v,itervp_CNN_1x64,win_size=WIN_SIZE),0,vp_init_model_loss)
vp_CNN_1x128_loss_vel   = np.insert(SSIM(true_v,itervp_CNN_1x128,win_size=WIN_SIZE),0,vp_init_model_loss)
vp_CNN_1x256_loss_vel   = np.insert(SSIM(true_v,itervp_CNN_1x256,win_size=WIN_SIZE),0,vp_init_model_loss)
vp_CNN_2x64_loss_vel    = np.insert(SSIM(true_v,itervp_CNN_2x64,win_size=WIN_SIZE),0,vp_init_model_loss)
vp_CNN_2x128_loss_vel   = np.insert(SSIM(true_v,itervp_CNN_2x128,win_size=WIN_SIZE),0,vp_init_model_loss)
vp_CNN_2x256_loss_vel   = np.insert(SSIM(true_v,itervp_CNN_2x256,win_size=WIN_SIZE),0,vp_init_model_loss)
vp_CNN_3x64_loss_vel    = np.insert(SSIM(true_v,itervp_CNN_3x64,win_size=WIN_SIZE),0,vp_init_model_loss)
vp_CNN_3x128_loss_vel   = np.insert(SSIM(true_v,itervp_CNN_3x128,win_size=WIN_SIZE),0,vp_init_model_loss)
vp_CNN_3x256_loss_vel   = np.insert(SSIM(true_v,itervp_CNN_3x256,win_size=WIN_SIZE),0,vp_init_model_loss)
ax4.plot(vp_baseline_loss, label="Baseline", color=colors["baseline"], linestyle='-', linewidth=2)
ax4.plot(vp_CNN_1x64_loss_vel, label="CNN-1x64", color=colors["CNN1x64"], linestyle="-", linewidth=2)
ax4.plot(vp_CNN_1x128_loss_vel, label="CNN-1x128", color=colors["CNN1x128"], linestyle="-", linewidth=2)
ax4.plot(vp_CNN_1x256_loss_vel, label="CNN-1x256", color=colors["CNN1x256"], linestyle="-", linewidth=2)
ax4.plot(vp_CNN_2x64_loss_vel, label="CNN-2x64", color=colors["CNN2x64"], linestyle="-", linewidth=2)
ax4.plot(vp_CNN_2x128_loss_vel, label="CNN-2x128", color=colors["CNN2x128"], linestyle="-", linewidth=2)
ax4.plot(vp_CNN_2x256_loss_vel, label="CNN-2x256", color=colors["CNN2x256"], linestyle="-", linewidth=2)
ax4.plot(vp_CNN_3x64_loss_vel, label="CNN-3x64", color=colors["CNN3x64"], linestyle="-", linewidth=2)
ax4.plot(vp_CNN_3x128_loss_vel, label="CNN-3x128", color=colors["CNN3x128"], linestyle="-", linewidth=2)
ax4.plot(vp_CNN_3x256_loss_vel, label="CNN-3x256", color=colors["CNN3x256"], linestyle="-", linewidth=2)

# Improve axis style for second subplot
ax4.set_xlabel('Iterations', fontsize=12)
ax4.set_ylabel('SSIM', fontsize=12)
# ax4.set_title("Model Residual", fontsize=12)
ax4.grid(True, linestyle='--', alpha=0.6)



# Adjust layout
plt.tight_layout()

# Show the plot

# plt.savefig("/ailab/user/liufeng1/project/04_Inversion/ADFWI-github/examples/dip/reparameterization-strategy/Marmousi2-nowater/cmp/Figures/Architecture_and_Strategy_Marmousi2_Misfit.png",bbox_inches='tight',dpi=300)
plt.show()

## Metrics

In [ ]:
print(
    f"{np.min(MAPE(true_v=true_v, inv_v=itervp_baseline)):.2f}",
    f"{np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x64)):.2f}",
    f"{np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x128)):.2f}",
    f"{np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x256)):.2f}",
    f"{np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x64)):.2f}",
    f"{np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x128)):.2f}",
    f"{np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x256)):.2f}",
    f"{np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x64)):.2f}",
    f"{np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x128)):.2f}",
    f"{np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x256)):.2f}"
)

window_size = 3
print(
    f"{np.max(SSIM(true_v=true_v, inv_v=itervp_baseline,win_size=window_size)):.2f}",
    f"{np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_1x64,win_size=window_size)):.2f}",
    f"{np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_1x128,win_size=window_size)):.2f}",
    f"{np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_1x256,win_size=window_size)):.2f}",
    f"{np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_2x64,win_size=window_size)):.2f}",
    f"{np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_2x128,win_size=window_size)):.2f}",
    f"{np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_2x256,win_size=window_size)):.2f}",
    f"{np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_3x64,win_size=window_size)):.2f}",
    f"{np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_3x128,win_size=window_size)):.2f}",
    f"{np.max(SSIM(true_v=true_v, inv_v=itervp_CNN_3x256,win_size=window_size)):.2f}"
)


print(
    f"{np.max(SNR(true_v=true_v, inv_v=itervp_baseline)):.2f}",
    f"{np.max(SNR(true_v=true_v, inv_v=itervp_CNN_1x64)):.2f}",
    f"{np.max(SNR(true_v=true_v, inv_v=itervp_CNN_1x128)):.2f}",
    f"{np.max(SNR(true_v=true_v, inv_v=itervp_CNN_1x256)):.2f}",
    f"{np.max(SNR(true_v=true_v, inv_v=itervp_CNN_2x64)):.2f}",
    f"{np.max(SNR(true_v=true_v, inv_v=itervp_CNN_2x128)):.2f}",
    f"{np.max(SNR(true_v=true_v, inv_v=itervp_CNN_2x256)):.2f}",
    f"{np.max(SNR(true_v=true_v, inv_v=itervp_CNN_3x64)):.2f}",
    f"{np.max(SNR(true_v=true_v, inv_v=itervp_CNN_3x128)):.2f}",
    f"{np.max(SNR(true_v=true_v, inv_v=itervp_CNN_3x256)):.2f}"
)

## Article Figure

In [ ]:
def plot_vel_single_for_all(fig,ax,v,title="",MSE="",vmin=None,vmax=None,cmap = "rainbow"):
    plt.rc('font',family='Times New Roman')
    # plm = ax.pcolormesh(x_mesh, z_mesh, v,cmap=cmap,vmin=vmin,vmax=vmax)
    x = np.arange(nx*3)*dx/3/1000
    z = np.arange(nz*3)*dz/3/1000
    x_mesh_new,z_mesh_new = np.meshgrid(x,z)

    v_new = griddata((x_mesh.flatten(), z_mesh.flatten()), v.flatten(), (x_mesh_new, z_mesh_new), method='cubic')
    
    plm = ax.pcolormesh(x_mesh_new, z_mesh_new, v_new,cmap=cmap,vmin=vmin,vmax=vmax,shading="nearest")
    ax.invert_yaxis()
    ax.tick_params(labelsize = 14)
    ax.set_title(title,fontsize=14)
    ax.text(0.2,0.4,MSE,fontsize=16,c="w")
    return plm

In [ ]:
WIN_SIZE=3
iterloss_ssim_baseline = SSIM(true_v=true_v, inv_v=itervp_baseline,win_size=WIN_SIZE)
iterloss_ssim_cnn_1x64 = SSIM(true_v=true_v, inv_v=itervp_CNN_1x64,win_size=WIN_SIZE)
iterloss_ssim_cnn_1x128 = SSIM(true_v=true_v, inv_v=itervp_CNN_1x128,win_size=WIN_SIZE)
iterloss_ssim_cnn_1x256 = SSIM(true_v=true_v, inv_v=itervp_CNN_1x256,win_size=WIN_SIZE)
iterloss_ssim_cnn_2x64 = SSIM(true_v=true_v, inv_v=itervp_CNN_2x64,win_size=WIN_SIZE)
iterloss_ssim_cnn_2x128 = SSIM(true_v=true_v, inv_v=itervp_CNN_2x128,win_size=WIN_SIZE)
iterloss_ssim_cnn_2x256 = SSIM(true_v=true_v, inv_v=itervp_CNN_2x256,win_size=WIN_SIZE)
iterloss_ssim_cnn_3x64 = SSIM(true_v=true_v, inv_v=itervp_CNN_3x64,win_size=WIN_SIZE)
iterloss_ssim_cnn_3x128 = SSIM(true_v=true_v, inv_v=itervp_CNN_3x128,win_size=WIN_SIZE)
iterloss_ssim_cnn_3x256 = SSIM(true_v=true_v, inv_v=itervp_CNN_3x256,win_size=WIN_SIZE)


In [ ]:
iterloss_mape_baseline = MAPE(true_v=true_v, inv_v=itervp_baseline)
iterloss_mape_cnn_1x64 = MAPE(true_v=true_v, inv_v=itervp_CNN_1x64)
iterloss_mape_cnn_1x128 = MAPE(true_v=true_v, inv_v=itervp_CNN_1x128)
iterloss_mape_cnn_1x256 = MAPE(true_v=true_v, inv_v=itervp_CNN_1x256)
iterloss_mape_cnn_2x64 = MAPE(true_v=true_v, inv_v=itervp_CNN_2x64)
iterloss_mape_cnn_2x128 = MAPE(true_v=true_v, inv_v=itervp_CNN_2x128)
iterloss_mape_cnn_2x256 = MAPE(true_v=true_v, inv_v=itervp_CNN_2x256)
iterloss_mape_cnn_3x64 = MAPE(true_v=true_v, inv_v=itervp_CNN_3x64)
iterloss_mape_cnn_3x128 = MAPE(true_v=true_v, inv_v=itervp_CNN_3x128)
iterloss_mape_cnn_3x256 = MAPE(true_v=true_v, inv_v=itervp_CNN_3x256)


In [ ]:
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.ticker import MaxNLocator
import numpy as np
plt.rcParams['svg.fonttype'] = 'none'

fig = plt.figure(figsize=(15,8))
gs = gridspec.GridSpec(2, 2, width_ratios=[1, 1], height_ratios=[1, 1], hspace=0.5)
ax01 = fig.add_subplot(gs[0, 0])
ax02 = fig.add_subplot(gs[0, 1])
ax11 = fig.add_subplot(gs[1, 0])
ax12 = fig.add_subplot(gs[1, 1])

# inversion result
im1 = plot_vel_single_for_all(fig,ax01,itervp_baseline[-1] ,title=None , MSE=r"MAPE:" + " " + r"{:.2f}".format(MAPE(true_v,itervp_baseline[-1])))
im2 = plot_vel_single_for_all(fig,ax02,itervp_CNN_2x128[-1],title=None , MSE=r"MAPE:" + " " + r"{:.2f}".format(MAPE(true_v,itervp_CNN_2x128[-1])))

ax01.set_xlabel("Distance (km)", fontsize=16)
ax02.set_xlabel("Distance (km)", fontsize=16)
ax01.set_ylabel("Depth (km)", fontsize=16)
ax02.set_ylabel("Depth (km)", fontsize=16)

ax01.xaxis.set_major_locator(MaxNLocator(7))
ax02.xaxis.set_major_locator(MaxNLocator(7))
ax01.set_title("Traditional FWI", fontsize=15, fontweight='bold')
ax02.set_title(r"DRFWI (CNN-$v_p$)", fontsize=15, fontweight='bold')

# Adjust layout for better spacing
plt.subplots_adjust(hspace=0.3, wspace=0.1)

import matplotlib as mpl
def add_bottom_cax(ax, pad, height,shrink=1):
    axpos = ax.get_position()
    width = axpos.x1 - axpos.x0
    left_position = axpos.x0 + width * (1 - shrink) / 2
    caxpos = mpl.transforms.Bbox.from_extents(
        left_position,
        axpos.y0 - pad,
        left_position + width * shrink,
        axpos.y0 - pad + height
    )
    cax = ax.figure.add_axes(caxpos)
    return cax

# add horizontal colorbar for [axes21, axes22]
cbar_ax1 = add_bottom_cax(ax01, 0.1, 0.02,shrink=0.8)
cbar1 = fig.colorbar(im1, cax=cbar_ax1, orientation='horizontal', pad=0.1, shrink=0.8)
cbar1.ax.tick_params(labelsize=14)
cbar1.ax.text(1.02, 0.5, r'$m/s$', fontsize=14, transform=cbar1.ax.transAxes, 
              verticalalignment='center', horizontalalignment='left')

cbar_ax2 = add_bottom_cax(ax02, 0.1, 0.02,shrink=0.8)
cbar2 = fig.colorbar(im2, cax=cbar_ax2, orientation='horizontal', pad=0.1, shrink=0.8)
cbar2.ax.tick_params(labelsize=14)
cbar2.ax.text(1.02, 0.5, r'$m/s$', fontsize=14, transform=cbar2.ax.transAxes, 
              verticalalignment='center', horizontalalignment='left')

# plot SSIM
cnn_losses_ssim = np.array([
    iterloss_ssim_cnn_2x64, iterloss_ssim_cnn_2x128, iterloss_ssim_cnn_2x256,
    iterloss_ssim_cnn_3x64, iterloss_ssim_cnn_3x128, iterloss_ssim_cnn_3x256
])
cnn_mean_ssim = np.mean(cnn_losses_ssim, axis=0)
cnn_std_ssim = np.std(cnn_losses_ssim, axis=0)

ax11.fill_between(np.arange(MAX_ITER), cnn_mean_ssim - cnn_std_ssim, cnn_mean_ssim + cnn_std_ssim, color="gray", alpha=0.3)
ax11.plot(np.arange(MAX_ITER), cnn_mean_ssim, color="crimson", linewidth=2, label="DRFWI")
ax11.plot(np.arange(MAX_ITER), iterloss_ssim_baseline, color="k", linewidth=2, label="Traditional FWI")
ax11.set_xlabel("Iteration", fontsize=16)
ax11.set_ylabel("SSIM", fontsize=16)
ax11.grid(alpha=0.3)
ax11.tick_params(labelsize=14)

# plot the MAPE
cnn_losses_mape = np.array([
    iterloss_mape_cnn_2x64, iterloss_mape_cnn_2x128, iterloss_mape_cnn_2x256,
    iterloss_mape_cnn_3x64, iterloss_mape_cnn_3x128, iterloss_mape_cnn_3x256
])
cnn_mean_mape = np.mean(cnn_losses_mape, axis=0)
cnn_std_mape = np.std(cnn_losses_mape, axis=0)

ax12.fill_between(np.arange(MAX_ITER), cnn_mean_mape - cnn_std_mape, cnn_mean_mape + cnn_std_mape, color="gray", alpha=0.3)
ax12.plot(np.arange(MAX_ITER), cnn_mean_mape, color="crimson", linewidth=2, label="DRFWI")
ax12.plot(np.arange(MAX_ITER), iterloss_mape_baseline, color="k", linewidth=2, label="Traditional FWI")
ax12.set_xlabel("Iteration", fontsize=16)
ax12.set_ylabel("MAPE", fontsize=16)
ax12.tick_params(labelsize=14)
ax12.grid(alpha=0.3)
ax12.legend(fontsize=16)

fig.text(0.090, 0.88, "(a)", fontsize=16, fontweight='bold', ha="center", va="center")
fig.text(0.505, 0.88, "(b)", fontsize=16, fontweight='bold', ha="center", va="center")
fig.text(0.090, 0.42, "(c)", fontsize=16, fontweight='bold', ha="center", va="center")
fig.text(0.505, 0.42, "(d)", fontsize=16, fontweight='bold', ha="center", va="center")

# plt.savefig("./Figures/FigureS2_RealStyle_Noise_InvertedResult.png",bbox_inches='tight',dpi=300)
plt.savefig("./Figures_SVG/FigureS2_RealStyle_Noise_InvertedResult.svg",bbox_inches='tight',format="svg")
plt.show()